<a href="https://colab.research.google.com/github/SohamDoesQuantum/MovieRecommendationSystem/blob/main/main_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:

movies = pd.read_csv('/content/drive/MyDrive/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/drive/MyDrive/tmdb_5000_credits.csv')

In [12]:
movies = movies.merge(credits,on='title')

In [15]:
movies= movies[[ 'id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [ ]:
movies.head()

In [17]:
import ast

In [19]:
def convert(text):
  L = []
  for i in ast.literal_eval(text):
    L.append(i['name'])
  return L

In [20]:
movies.dropna(inplace=True)

In [ ]:
movies['genres'] = movies['genres'].apply(convert)
movies.head()

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)
movies.head()

In [23]:
def convert3(text):
  L =[]
  counter = 0
  for i in ast.literal_eval(text):
    if counter < 3:
      L.append(i['name'])
    counter+=1
  return L

In [ ]:
movies['cast'] = movies['cast'].apply(convert)
movies.head()

In [25]:
def get_director(text):
  L = []
  for i in ast.literal_eval(text):
    if i['job'] == 'Director':
      L.append(i['name'])
      break
  return L

In [26]:
movies['crew'] = movies['crew'].apply(get_director)

In [31]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [34]:
movies['tag'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
new_df = movies[['id','title','tag']].copy()
new_df['tag'] = new_df['tag'].apply(lambda x:" ".join(x))
new_df['tag'] = new_df['tag'].apply(lambda x:x.lower())

In [35]:
new_df.head()

,id,title,tag
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [ ]:
!pip install nltk

In [37]:
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [38]:
def stem(text):
  y =[]
  for i in text.split():
    y.append(ps.stem(i))
  return " ".join(y)

In [41]:
new_df['tag'] = new_df['tag'].apply(stem)
new_df['tag']

,tag
0,"in the 22nd century, a parapleg marin is dispa..."
1,"captain barbossa, long believ to be dead, ha c..."
2,a cryptic messag from bond’ past send him on a...
3,follow the death of district attorney harvey d...
4,"john carter is a war-weary, former militari ca..."
...,...
4804,el mariachi just want to play hi guitar and ca...
4805,a newlyw couple' honeymoon is upend by the arr...
4806,"""signed, sealed, delivered"" introduc a dedic q..."
4807,when ambiti new york attorney sam is sent to s...


In [49]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words = 'english')

In [50]:
vectors =  cv.fit_transform(new_df['tag']).toarray()

In [51]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [60]:
def recommend(movie):
  movie_index = new_df[new_df['title'] == movie].index[0]
  distances = similarity[movie_index]
  movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
  movies_list = movies_list[1:6]
  for i in movies_list:
    print(new_df.iloc[i[0]].title)
recommend('Batman Begins')

Batman
The Dark Knight Rises
Batman
Rockaway


In [61]:
import pickle
from google.colab import files


pickle.dump(new_df.to_dict(), open('movies_dict.pkl', 'wb'))


pickle.dump(similarity, open('similarity.pkl', 'wb'))


files.download('movies_dict.pkl')
files.download('similarity.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>